# Real atmospheric and ocean forcing

**Learning goals:** Inspect real ERA5 and HYCOM data and connect them to SCHISM forcing objects.

**Prerequisites:** Lesson 3; access to the shared `rom-py/rompy-test-data` fixture bundle.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_03_schism_grid_data](../journey_03_schism_grid_data/)

Next: [journey_05_schism_boundaries](../journey_05_schism_boundaries/)


## Why this matters: SCHISM data preparation

**Without Rompy:** preparing HYCOM boundary conditions can mean downloading a large global dataset, selecting the run period and region, interpolating onto open-boundary nodes, extracting the required variables, and writing `elev2D.th.nc` or other SCHISM files. ERA5 and tidal inputs require similarly separate preparation steps.

**With Rompy:** source objects, grid metadata, time ranges, filters, and boundary mappings are assembled into `SCHISMConfig`. Workspace generation carries out the configured cropping, interpolation, boundary extraction, and SCHISM-format conversion. The modeller still chooses appropriate datasets, variables, coordinates, numerical settings, and scientific validation checks.

The following cells show the source fields, model domain, and generated artefacts so this automation remains inspectable.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


In [ ]:
import xarray as xr

# `case` is supplied by the previous lesson when running sequentially.
era5 = xr.open_dataset(case / "era5.nc")
hycom = xr.open_dataset(case / "hycom.nc")
print("ERA5 variables:", list(era5.data_vars))
print("HYCOM variables:", list(hycom.data_vars))
era5.close()
hycom.close()


## Visual verification: source fields

These are real cropped ERA5 and HYCOM fixtures. Plotting them before generation checks variable names, coordinate orientation, time coverage, and the spatial relationship to the regional mesh.


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

era5 = xr.open_dataset(case / "era5.nc")
hycom = xr.open_dataset(case / "hycom.nc")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
era5.u10.isel(time=0).plot(ax=axes[0], cmap="coolwarm")
axes[0].set_title("ERA5 eastward wind")
hycom.surf_el.isel(time=0).plot(ax=axes[1], cmap="BrBG")
axes[1].set_title("HYCOM sea-surface elevation")
plt.show()
era5.close(); hycom.close()
